In [ ]:
for p in sorted(glob.glob(f'{FEAT_DIR}/*.npz')):
    z = np.load(p)
    k0 = z.files[0]
    print(f'{os.path.basename(p):30s}  n={len(z.files):4d}  dim={z[k0].shape}')

## Index of saved features

Later notebooks use this list to know which feature banks exist.

In [ ]:
def lowlevel_feature(vid, n_mel=8):
    # vision summaries over cached frames
    paths = sorted(glob.glob(f'{frame_dir(vid)}/f*.jpg'))
    grays = [cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2GRAY).astype(np.float32) / 255. for p in paths]
    grays = np.stack(grays)
    lum_mean = grays.mean((1, 2))
    lum_std  = grays.std((1, 2))
    motion   = np.abs(np.diff(grays, axis=0)).mean((1, 2))

    # audio summaries
    y, sr = sf.read(audio_path(vid))
    y = y.astype(np.float32) / max(np.abs(y).max(), 1.0)
    rms  = float(np.sqrt((y**2).mean()))
    zcr  = float(((y[:-1] * y[1:]) < 0).mean())
    mel  = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mel, fmax=sr//2)
    mel  = np.log1p(mel).mean(axis=1)

    return np.concatenate([
        [lum_mean.mean(), lum_std.mean(), motion.mean(), motion.std()],
        lum_mean, lum_std, motion if motion.size else [0.0],
        [rms, zcr], mel,
    ]).astype(np.float32)

feats = {}
for v in tqdm(ALL_IDS, desc='lowlevel'):
    feats[int(v)] = lowlevel_feature(v)
# pad to common length (motion length depends on N_FRAMES-1)
L = max(f.size for f in feats.values())
for k, v in feats.items():
    if v.size < L:
        feats[k] = np.pad(v, (0, L - v.size))
print(save_features('lowlevel', feats), 'dim', L)

## Low-level features

Per video we store summaries that are cheap, interpretable and useful as a control band in the encoding model:
- luminance mean and std over frames
- motion energy = mean |frame_t - frame_{t-1}| (grayscale)
- audio envelope: RMS, zero-crossing-rate, log mel-energy at a few bands

All concatenated into a single feature vector per video.

In [ ]:
from transformers import ClapModel, ClapProcessor
import librosa

CLAP_CKPT = 'laion/clap-htsat-unfused'
clap_proc  = ClapProcessor.from_pretrained(CLAP_CKPT)
clap_model = ClapModel.from_pretrained(CLAP_CKPT).to(DEVICE).eval()
CLAP_SR    = clap_proc.feature_extractor.sampling_rate   # 48 kHz

@torch.no_grad()
def clap_feature(vid):
    y, sr = sf.read(audio_path(vid))
    y = y.astype(np.float32) / max(np.abs(y).max(), 1.0)
    if sr != CLAP_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=CLAP_SR)
    inp = clap_proc(audios=y, sampling_rate=CLAP_SR, return_tensors='pt').to(DEVICE)
    f = clap_model.get_audio_features(**inp).float().cpu().squeeze(0).numpy()
    return f / max(np.linalg.norm(f), 1e-6)

feats = {}
for v in tqdm(ALL_IDS, desc='clap'):
    feats[int(v)] = clap_feature(v)
print(save_features('clap', feats), feats[int(ALL_IDS[0])].shape)

## CLAP audio embedding

CLAP (HuggingFace `laion/clap-htsat-unfused`) is used as the audio-side feature because BEATs has no pinned HF checkpoint. CLAP's text-aligned audio space also makes it useful for the contrastive decoder in 05.

In [ ]:
from transformers import VideoMAEImageProcessor, VideoMAEModel

VIDMAE_CKPT = 'MCG-NJU/videomae-large'
vm_proc  = VideoMAEImageProcessor.from_pretrained(VIDMAE_CKPT)
vm_model = VideoMAEModel.from_pretrained(VIDMAE_CKPT).to(DEVICE).eval()

def read_video_frames(vid, n=16, size=224):
    cap = cv2.VideoCapture(video_path(vid))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    picks = np.linspace(0, max(total - 1, 0), n).astype(int)
    frames = []
    for fno in picks:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(fno))
        ok, f = cap.read()
        if not ok:
            f = np.zeros((size, size, 3), dtype=np.uint8)
        else:
            f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        frames.append(f)
    cap.release()
    return frames  # list of H×W×3 uint8

@torch.no_grad()
def videomae_feature(vid):
    frames = read_video_frames(vid, n=16)
    inputs = vm_proc(frames, return_tensors='pt').to(DEVICE)
    out = vm_model(**inputs).last_hidden_state   # (1, T*P, D)
    return out.mean(1).squeeze(0).float().cpu().numpy()

feats = {}
for v in tqdm(ALL_IDS, desc='videomae'):
    feats[int(v)] = videomae_feature(v)
print(save_features('videomae_v2', feats), feats[int(ALL_IDS[0])].shape)

## VideoMAE-v2 clip embedding

Reads the 16 frames straight from the mp4 (bypasses the jpg cache so the temporal signal survives). The HF `MCG-NJU/videomae-large` checkpoint is used as a VideoMAE-v2 proxy; swap in V-JEPA 2 by changing `VIDMAE_CKPT` and the processor.

In [ ]:
import open_clip
from PIL import Image

clip_model, _, clip_pre = open_clip.create_model_and_transforms(
    'ViT-L-14', pretrained='openai', device=DEVICE)
clip_model.eval()

@torch.no_grad()
def clip_video_feature(vid, batch=16):
    paths = sorted(glob.glob(f'{frame_dir(vid)}/f*.jpg'))
    imgs = torch.stack([clip_pre(Image.open(p).convert('RGB')) for p in paths]).to(DEVICE)
    feats = []
    for i in range(0, len(imgs), batch):
        feats.append(clip_model.encode_image(imgs[i:i+batch]).float().cpu())
    f = torch.cat(feats, 0)
    f = f / f.norm(dim=-1, keepdim=True).clamp_min(1e-6)
    return f.mean(0).numpy()

feats = {}
for v in tqdm(ALL_IDS, desc='clip'):
    feats[int(v)] = clip_video_feature(v)
print(save_features('clip_vitl14', feats), feats[int(ALL_IDS[0])].shape)

## DINOv2 ViT-L/14 per frame (frame-averaged)

Pure self-supervised vision model — complements CLIP's semantic signal with a ventral-stream-leaning representation (strong on NSD / Algonauts encoding).

In [ ]:
from transformers import AutoImageProcessor, AutoModel

DINO_CKPT  = 'facebook/dinov2-large'
dino_proc  = AutoImageProcessor.from_pretrained(DINO_CKPT)
dino_model = AutoModel.from_pretrained(DINO_CKPT).to(DEVICE).eval()

@torch.no_grad()
def dinov2_video_feature(vid, batch=16):
    paths = sorted(glob.glob(f'{frame_dir(vid)}/f*.jpg'))
    imgs = [Image.open(p).convert('RGB') for p in paths]
    feats = []
    for i in range(0, len(imgs), batch):
        inp = dino_proc(imgs[i:i+batch], return_tensors='pt').to(DEVICE)
        out = dino_model(**inp).last_hidden_state      # (b, 1+patches, D)
        feats.append(out[:, 0].float().cpu())          # CLS token
    f = torch.cat(feats, 0)
    f = f / f.norm(dim=-1, keepdim=True).clamp_min(1e-6)
    return f.mean(0).numpy()

feats = {}
for v in tqdm(ALL_IDS, desc='dinov2'):
    feats[int(v)] = dinov2_video_feature(v)
print(save_features('dinov2_vitl14', feats), feats[int(ALL_IDS[0])].shape)

## CLIP ViT-L/14 per frame (frame-averaged)

In [ ]:
for layer in ALEX_LAYERS:
    z = np.load(f'{ALEX_WORK}/consolidated_{layer}.npz')
    feats = {}
    for v in ALL_IDS:
        keys = [k for k in z.files if k.startswith(f'{int(v):05d}_')]
        assert keys, f'no features for {v} / {layer}'
        feats[int(v)] = np.mean([z[k].flatten() for k in keys], axis=0)
    path = save_features(f'alexnet_{layer}', feats)
    print(layer, '->', path, feats[int(ALL_IDS[0])].shape)

In [ ]:
from net2brain.feature_extraction import FeatureExtractor

ALEX_LAYERS = ['features.0', 'features.3', 'features.6', 'features.8', 'features.10']
ALEX_WORK   = f'{FEAT_DIR}/_alexnet_work'
os.makedirs(ALEX_WORK, exist_ok=True)

# Net2Brain expects a flat folder of images; symlink all frames with a vid_frame.jpg name.
ALEX_FRAMES = f'{FEAT_DIR}/_alexnet_frames'
os.makedirs(ALEX_FRAMES, exist_ok=True)
for v in ALL_IDS:
    for p in sorted(glob.glob(f'{frame_dir(v)}/f*.jpg')):
        link = f'{ALEX_FRAMES}/{int(v):05d}_' + os.path.basename(p)
        if not os.path.exists(link):
            os.symlink(os.path.abspath(p), link)

fx = FeatureExtractor(model='AlexNet', netset='Standard', device=DEVICE)
fx.extract(data_path=ALEX_FRAMES, layers_to_extract=ALEX_LAYERS, save_path=ALEX_WORK)

## AlexNet (multi-layer) via Net2Brain

Features from several AlexNet conv/fc layers (baseline used only `features.3`). We frame-average per video, keeping each layer separate so 04 can band them.

In [ ]:
def save_features(name, feats):
    """feats: dict {vid:int -> np.ndarray}. Writes features/<name>.npz."""
    path = f'{FEAT_DIR}/{name}.npz'
    tmp = path + '.tmp.npz'
    np.savez_compressed(tmp, **{f'{int(k):04d}': np.asarray(v, dtype=np.float32) for k, v in feats.items()})
    os.replace(tmp, path)
    return path

def load_features(name):
    z = np.load(f'{FEAT_DIR}/{name}.npz')
    return {int(k): z[k] for k in z.files}

## Shared save helper

Each model writes a single `features/<name>.npz` with keys `"0001".."1102"`. Later notebooks load it as `z[str(int(vid))]`.

In [ ]:
import av
import soundfile as sf

def audio_path(vid): return f'{AUDIO_DIR}/{int(vid):05d}.wav'

def dump_audio(ids, sr=AUDIO_SR):
    """Decode mono audio with PyAV, resample to `sr`, write 16-bit WAV. Silent track -> zeros."""
    for sid in tqdm(ids, desc='audio'):
        out = audio_path(sid)
        if os.path.exists(out):
            continue
        try:
            with av.open(video_path(sid)) as container:
                astreams = [s for s in container.streams if s.type == 'audio']
                if not astreams:
                    sf.write(out, np.zeros(sr * 3, dtype=np.int16), sr)
                    continue
                stream = astreams[0]
                resampler = av.AudioResampler(format='s16', layout='mono', rate=sr)
                chunks = []
                for frame in container.decode(stream):
                    for rf in resampler.resample(frame):
                        chunks.append(rf.to_ndarray().reshape(-1))
                data = np.concatenate(chunks) if chunks else np.zeros(sr * 3, dtype=np.int16)
                sf.write(out, data.astype(np.int16), sr)
        except Exception as e:
            print('audio decode failed for', sid, e)
            sf.write(out, np.zeros(sr * 3, dtype=np.int16), sr)

dump_audio(ALL_IDS)
print('audio files written:', len(glob.glob(f'{AUDIO_DIR}/*.wav')))

In [ ]:
import cv2

def frame_dir(vid): return f'{FRAME_DIR}/{int(vid):05d}'

def dump_frames(ids, n_frames=N_FRAMES):
    for sid in tqdm(ids, desc='frames'):
        out = frame_dir(sid); os.makedirs(out, exist_ok=True)
        done = sorted(glob.glob(f'{out}/f*.jpg'))
        if len(done) == n_frames:
            continue
        cap = cv2.VideoCapture(video_path(sid))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        picks = np.linspace(0, max(total - 1, 0), n_frames).astype(int)
        for k, fno in enumerate(picks):
            path = f'{out}/f{k:02d}.jpg'
            if os.path.exists(path):
                continue
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(fno))
            ok, frame = cap.read()
            if ok:
                cv2.imwrite(path, frame)
        cap.release()

dump_frames(ALL_IDS)
print('frames per video (first 3):',
      [len(glob.glob(f'{frame_dir(v)}/f*.jpg')) for v in ALL_IDS[:3]])

## Decode frames and audio

`dump_frames` is the baseline logic extended to `N_FRAMES=16`. `dump_audio` extracts the audio track with PyAV, resamples to 16 kHz mono and writes a WAV. Both are idempotent — rerunning is cheap.

In [ ]:
def video_path(vid):
    vid = int(vid)
    d = STIM_TRAIN_DIR if vid <= 1000 else STIM_TEST_DIR
    return f'{d}/{vid:04d}.mp4'

missing = [v for v in ALL_IDS if not os.path.exists(video_path(v))]
assert not missing, f'missing videos: {missing[:10]}'
print(len(ALL_IDS), 'videos available')

## Video id → file path

Train videos are `stimuli_train/0001.mp4..1000.mp4` (ids 1..1000), test videos are `stimuli_test/1001.mp4..1102.mp4` (ids 1001..1102).

In [ ]:
STIM_TRAIN_DIR = './stimuli_train'
STIM_TEST_DIR  = './stimuli_test'
FRAME_DIR      = './frames'            # jpg frames per video
AUDIO_DIR      = './audio'             # 16 kHz mono wavs per video
FEAT_DIR       = './features'          # final per-model npz files
os.makedirs(FRAME_DIR, exist_ok=True)
os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(FEAT_DIR, exist_ok=True)

N_FRAMES   = 16
AUDIO_SR   = 16_000
TRAIN_IDS  = np.arange(1,    1001)
TEST_IDS   = np.arange(1001, 1103)
ALL_IDS    = np.concatenate([TRAIN_IDS, TEST_IDS])

In [ ]:
import os, glob, io
import numpy as np
import torch
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

In [ ]:
# Install dependencies.
# Local (krabiki conda env, Python 3.11):
# %pip install -r requirements.txt
# %pip install "net2brain @ git+https://github.com/cvai-roig-lab/Net2Brain.git"
#
# Kaggle GPU kernel: torch/numpy/scipy/sklearn/cv2/PIL/tqdm are preinstalled; only install extras.
# %pip install -q av soundfile librosa mne himalaya open_clip_torch transformers
# %pip install -q "net2brain @ git+https://github.com/cvai-roig-lab/Net2Brain.git"

# 03 — Feature extraction (per video, GPU)

Extracts per-video features used by the encoding model (04) and the contrastive decoder (05). All features are keyed by **video id** (train 1–1000, test 1001–1102) so they are shared across subjects.

Outputs (all under `./features/`, git-ignored):
- `alexnet_<layer>.npz`      — multiple AlexNet layers via Net2Brain, frame-averaged
- `clip_vitl14.npz`          — CLIP ViT-L/14 image embeddings, frame-averaged
- `videomae_v2.npz`          — VideoMAE-v2 clip embedding (fallback: V-JEPA 2)
- `beats.npz` / `clap.npz`   — audio embeddings (BEATs preferred, CLAP fallback)
- `lowlevel.npz`             — luminance, motion energy, audio envelope summaries

Design notes:
- Frames: 16 uniformly-spaced frames per video (baseline uses 8; we use 16 for finer time resolution).
- Audio: decoded with PyAV at 16 kHz mono.
- Each model block is independent; comment out what you don't need. Cached files are not recomputed.
- Helpers from 00 are duplicated here so the notebook is standalone.